# [Project] Bangladesh Agriculture Data Cleaning Pipeline
**Mục tiêu**: Làm sạch toàn bộ dữ liệu nông nghiệp Bangladesh một cách hệ thống, giữ nguyên vẹn tất cả các thuộc tính (features) quan trọng nhưng tối ưu hóa mã nguồn chạy tinh gọn, rõ ràng giống như một dự án Weather Analyst chuyên nghiệp.

## 1. Khởi tạo và Đọc dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import calendar

# Đọc dữ liệu từ file gốc
df = pd.read_csv('Bangladesh_database_Final_Merged.csv')
df = df.drop_duplicates()
df.columns = df.columns.str.strip()
print(f"Kích thước dữ liệu ban đầu: {df.shape}")

## 2. Lọc các bản ghi lỗi nghiêm trọng
Loại bỏ các dòng có dữ liệu phi lý (Diện tích hoặc Sản lượng <= 0) hoặc mã lỗi hệ thống (`#Ref!`).

In [ ]:
# Loại bỏ các dòng có Production hoặc Area không hợp lệ
df = df[(df['Production'] > 0) & (df['Area'] > 0)]

# Loại bỏ dữ liệu nhiễu ở cột Crop Name
if 'Crop Name' in df.columns:
    df = df[df['Crop Name'].str.strip() != '#Ref!']

print(f"Kích thước sau khi lọc hàng lỗi: {df.shape}")

## 3. Chuẩn hóa Định dạng và Sửa lỗi Văn bản (Text Cleaning)
Đồng nhất kiểu chữ viết hoa đầu từ (`Title Case`), dọn dẹp khoảng trắng thừa, xóa chú thích tiếng Việt trong ngoặc và chuẩn hóa lịch mùa vụ.

In [ ]:
# 1. Làm sạch khoảng trắng và định dạng Title Case cho toàn bộ cột dạng chữ
str_cols = df.select_dtypes(include=['object']).columns
for col in str_cols:
    df[col] = df[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True).str.title()

# 2. Loại bỏ phần giải nghĩa tiếng Việt nằm trong dấu ngoặc đơn
for col in ['pH_Suitability', 'Dominant_Soil_Texture']:
    if col in df.columns:
        df[col] = df[col].str.replace(r'\s*\(.*\)', '', regex=True)

# 3. Chuẩn hóa tên tháng viết tắt và sửa lỗi Typo lịch mùa vụ
month_map = {'Jan': 'January', 'Feb': 'February', 'Aug': 'August', 'Sep': 'September', 'Oct': 'October', 'Nov': 'November', 'Dec': 'December'}
df['Transplant'] = df['Transplant'].replace(month_map)
df.replace({'Throuout The Year': 'Throughout The Year'}, inplace=True)

# 4. Tự động điền thông tin lịch mùa vụ bị thiếu dựa theo quy luật của từng Season
season_rules = {
    'Kharif 1': {'Transplant': 'March', 'Growth': 'April To May', 'Harvest': 'June'},
    'Kharif 2': {'Transplant': 'July', 'Growth': 'August To October', 'Harvest': 'November'},
    'Rabi': {'Transplant': 'November', 'Growth': 'December To February', 'Harvest': 'March'}
}
cycle_cols = ['Transplant', 'Growth', 'Harvest']
invalid_vals = ['No Need To Do', 'Throughout The Year']

for season, rules in season_rules.items():
    season_mask = df['Season'] == season
    for col in cycle_cols:
        invalid_mask = df[col].isin(invalid_vals) | df[col].isna()
        df.loc[season_mask & invalid_mask, col] = rules[col]

print("Hoàn thành chuẩn hóa các cột văn bản.")

## 4. Xử lý logic và hiệu chỉnh các chỉ số kỹ thuật
Ép kiểu dữ liệu số, hiệu chỉnh tỷ lệ các chỉ số viễn thám ($FPAR$) và nhiệt độ bề mặt đất ($LST$), cân bằng thành phần kết cấu đất xấp xỉ 100%.

In [ ]:
# 1. Ép kiểu số an toàn cho các cột đo lường khí hậu
numeric_fix_cols = ['Avg Temp', 'Avg Humidity', 'Max Temp', 'Min Temp', 'Rainfall', 'Wind_Mean', 'pH']
for col in numeric_fix_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].mean())

# 2. Hiệu chỉnh hệ số tỷ lệ cho FPAR
if 'FPAR' in df.columns:
    df['FPAR'] = df['FPAR'] * 0.1

# 3. Chuyển đổi LST từ Kelvin sang độ C
if 'LST_Kelvin' in df.columns:
    df['LST_C'] = df['LST_Kelvin'] - 273.15
    df = df.drop(columns=['LST_Kelvin'])

# 4. Chuẩn hóa tổng tỷ lệ thành phần đất (Clay + Sand + Silt = 100%)
soil_parts = ['Clay', 'Sand', 'Silt']
if all(col in df.columns for col in soil_parts):
    soil_sum = df[soil_parts].sum(axis=1)
    df[soil_parts] = df[soil_parts].div(soil_sum, axis=0) * 100

# 5. Đảo ngược lỗi hoán đổi cột độ ẩm nếu có
if 'Max Relative Humidity' in df.columns and 'Min Relative Humidity' in df.columns:
    # Đảm bảo Max luôn lớn hơn hoặc bằng Min
    mask = df['Max Relative Humidity'] < df['Min Relative Humidity']
    df.loc[mask, ['Min Relative Humidity', 'Max Relative Humidity']] = df.loc[mask, ['Max Relative Humidity', 'Min Relative Humidity']].values

print("Hoàn thành hiệu chỉnh logic và chỉ số vật lý.")

## 5. Xử lý Giá trị bất thường (Outliers & Ngưỡng thực tế)
Sử dụng phương pháp giới hạn biên (Capping) bằng phân vị IQR và gắn nhãn (Flagging) đối với các sự kiện khí hậu cực đoan.

In [ ]:
# Giới hạn ngưỡng thực tế cho một số cột cốt lõi
valid_ranges = {
    'Avg Temp': (5, 50), 'pH': (3.5, 9.5), 'Avg Humidity': (10, 100), 'Rainfall': (0, 5000)
}
for col, (min_val, max_val) in valid_ranges.items():
    if col in df.columns:
        df[col] = np.clip(df[col], min_val, max_val)

# Hàm xử lý Outliers nâng cao bằng IQR
def advanced_outlier_cleaner(dataframe, capping_cols, extreme_cols):
    dff = dataframe.copy()
    # Gắn nhãn + Capping biến cực đoan thời tiết
    for col in extreme_cols:
        if col in dff.columns:
            Q1, Q3 = dff[col].quantile(0.25), dff[col].quantile(0.75)
            upper_bound = Q3 + 1.5 * (Q3 - Q1)
            dff[f'is_extreme_{col}'] = (dff[col] > upper_bound).astype(int)
            dff[col] = np.clip(dff[col], None, upper_bound)
 
    # Capping biến chỉ số sinh trưởng và đất
    for col in capping_cols:
        if col in dff.columns:
            Q1, Q3 = dff[col].quantile(0.25), dff[col].quantile(0.75)
            IQR = Q3 - Q1
            dff[col] = np.clip(dff[col], Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    return dff

capping_list = ['Nitrogen', 'LAI', 'Organic_Carbon', 'Max Relative Humidity', 'Min Relative Humidity', 'Avg Humidity', 'EVI', 'FPAR', 'NDVI_Season_Mean', 'NDVI_Season_Min']
extreme_list = ['Heat_Stress_Days', 'Wind_Max']

df = advanced_outlier_cleaner(df, capping_list, extreme_list)
print("Hoàn thành xử lý giá trị ngoại lai bằng IQR.")

## 6. Sửa lỗi logic Sản lượng & Tính toán Năng suất (Yield)
Sửa lỗi hệ thống nhân sai đơn vị 100 lần của quả Mít (Jack Fruit) và tính toán thuộc tính Năng suất (`Yield`).

In [ ]:
# Tính toán lại Yield chuẩn ban đầu
df['Yield'] = df['Production'] / df['Area']

# Phát hiện lỗi nhân 100 lần của quả Mít (Jack Fruit) qua phân vị IQR
jf_data = df[df['Crop Name'] == 'Jack Fruit']
Q1, Q3 = jf_data['Yield'].quantile(0.25), jf_data['Yield'].quantile(0.75)
jf_upper_bound = Q3 + 1.5 * (Q3 - Q1)

outlier_condition = (df['Crop Name'] == 'Jack Fruit') & (df['Yield'] > jf_upper_bound)
print(f"Số lượng dòng lỗi Jack Fruit phát hiện: {outlier_condition.sum()}")

# Sửa lỗi: Chia Production cho 100 và cập nhật lại Yield
df.loc[outlier_condition, 'Production'] = df.loc[outlier_condition, 'Production'] / 100
df['Yield'] = df['Production'] / df['Area']

# Xóa bỏ các cột nháp tạm thời nếu có
df = df.drop(columns=['AP Ratio'], errors='ignore')
print(f"Kích thước file sạch cuối cùng: {df.shape}")

## 7. Xuất file dữ liệu sạch hoàn chỉnh

In [ ]:
output_file = 'Agri_Data_Cleaned.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"🎉 Thành công! File dữ liệu sạch giữ nguyên toàn bộ các feature đã được xuất tại: {output_file}")